Criar uma tabela de ordens ja com:<br>
quantidade de itens, vendendores, lojas, mapeamento de Status e valor total da venda (valor total - desconto)<br><br>
Mapeamento Status:<br>
1 THEN 'Pending'<br>
2 THEN 'Processing'<br>
3 THEN 'Shipped'<br>
4 THEN 'Delivered'<br>
ELSE 'Unknown'<br>

In [0]:
%python
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/resource/bronze/'
silver_path   = '/Volumes/bikestore/resource/silver/'
gold_path     = '/Volumes/bikestore/resource/gold/'
resource_path = '/Workspace/Users/wesllan2000@yahoo.com.br/project_databricks_bikestore/resource/origem'


In [0]:
%python
#criar dicionario path
bronze_map={
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    'tmp_bronze_customers' : f'{bronze_path}/customers/',
    'tmp_bronze_order_items' : f'{bronze_path}/order_items/',
    'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    'tmp_bronze_stores' : f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (
        spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from tmp_bronze_orders limit 10;

In [0]:
%sql
describe tmp_bronze_orders

In [0]:
%sql
describe tmp_bronze_stores

In [0]:
select * from tmp_bronze_order_items limit 10;

In [0]:
describe tmp_bronze_order_items

In [0]:
SELECT
    oi.order_id
    ,oi.item_id
    ,oi.product_id
    ,oi.quantity
    ,oi.list_price
    ,oi.discount
    ,ROUND((oi.list_price * oi.quantity) * (1 - oi.discount),2) AS total_sale
  FROM tmp_bronze_order_items AS oi

In [0]:
%sql
WITH order_items AS(
  SELECT
    oi.order_id
    ,oi.item_id
    ,oi.product_id
    ,oi.quantity
    ,oi.list_price
    ,oi.discount
    ,ROUND((oi.list_price * oi.quantity) * (1 - oi.discount),2) AS total_sale
  FROM tmp_bronze_order_items AS oi
    
)
SELECT
  o.order_id,
  o.customer_id,
  --o.order_status,
    CASE
      WHEN o.order_status = 1 THEN 'Pending'
      WHEN o.order_status = 2 THEN 'Processing'
      WHEN o.order_status = 3 THEN 'Shipped'
      WHEN o.order_status = 4 THEN 'Delivered'
      ELSE'Unknown'
    END AS order_status,
  sto.store_name,
  sto.city,
  sto.state,
  sta.first_name,
  sta.email,
  sta.active,
  o.order_date,
  o.required_date,
  o.shipped_date,
  oit.product_id,
  oit.quantity,
  oit.total_sale,
  oit.list_price,
  oit.discount
FROM
  tmp_bronze_orders AS o
LEFT JOIN tmp_bronze_stores AS sto ON o.store_id = sto.store_id
LEFT JOIN tmp_bronze_staffs AS sta ON o.staff_id = sta.staff_id
LEFT JOIN order_items AS oit ON o.order_id = oit.order_id

In [0]:
%python
df_orders_silver = spark.sql("""
WITH order_items AS(
  SELECT
    oi.order_id
    ,oi.item_id
    ,oi.product_id
    ,oi.quantity
    ,oi.list_price
    ,oi.discount
    ,ROUND((oi.list_price * oi.quantity) * (1 - oi.discount),2) AS total_sale
  FROM tmp_bronze_order_items AS oi
    
)
SELECT
  o.order_id,
  o.customer_id,
  --o.order_status,
    CASE
      WHEN o.order_status = 1 THEN 'Pending'
      WHEN o.order_status = 2 THEN 'Processing'
      WHEN o.order_status = 3 THEN 'Shipped'
      WHEN o.order_status = 4 THEN 'Delivered'
      ELSE'Unknown'
    END AS order_status,
  sto.store_name,
  sto.city,
  sto.state,
  sta.first_name,
  sta.email,
  sta.active,
  o.order_date,
  o.required_date,
  o.shipped_date,
  oit.product_id,
  oit.quantity,
  oit.total_sale,
  oit.list_price,
  oit.discount
FROM
  tmp_bronze_orders AS o
LEFT JOIN tmp_bronze_stores AS sto ON o.store_id = sto.store_id
LEFT JOIN tmp_bronze_staffs AS sta ON o.staff_id = sta.staff_id
LEFT JOIN order_items AS oit ON o.order_id = oit.order_id
"""
)

# Salvar em Delta na silver
df_orders_silver.write\
  .mode("overwrite")\
  .format('delta')\
  .option('mergeSchema', 'true')\
  .save(f'{silver_path}/orders')

In [0]:
%sql
--criar tabela na silver em bikestore.logistics
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_orders 
AS (
WITH order_items AS(
  SELECT
    oi.order_id
    ,oi.item_id
    ,oi.product_id
    ,oi.quantity
    ,oi.list_price
    ,oi.discount
    ,ROUND((oi.list_price * oi.quantity) * (1 - oi.discount),2) AS total_sale
  FROM tmp_bronze_order_items AS oi
    
)
SELECT
  o.order_id,
  o.customer_id,
  --o.order_status,
    CASE
      WHEN o.order_status = 1 THEN 'Pending'
      WHEN o.order_status = 2 THEN 'Processing'
      WHEN o.order_status = 3 THEN 'Shipped'
      WHEN o.order_status = 4 THEN 'Delivered'
      ELSE'Unknown'
    END AS order_status,
  sto.store_name,
  sto.city,
  sto.state,
  sta.first_name,
  sta.email,
  sta.active,
  o.order_date,
  o.required_date,
  o.shipped_date,
  oit.product_id,
  oit.quantity,
  oit.total_sale,
  oit.list_price,
  oit.discount
FROM
  tmp_bronze_orders AS o
LEFT JOIN tmp_bronze_stores AS sto ON o.store_id = sto.store_id
LEFT JOIN tmp_bronze_staffs AS sta ON o.staff_id = sta.staff_id
LEFT JOIN order_items AS oit ON o.order_id = oit.order_id
)



In [0]:
%sql
select * from bikestore.logistics.silver_orders limit 10


In [0]:
%skip
%python
/*Exemplo com LOCATION onde o storage é externo*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_products
LOCATION 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/bikestore/silver/product';

